In [ ]:
"""App4 (CV→DV State Transfer) benchmark analysis.

Reads .out log files from  raw_data/perlmutter_gpu/app4/runs/<campaign>/
produced by runner_gpu.sh (benchmark_id=14) or the bench_*.py scripts.
Produces comparison bar plots in the same style as qcvdv_results_latest.ipynb.
"""
import glob, os, re
from pathlib import Path

import matplotlib as mpl
import matplotlib.colors as mcolors
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.patches import Patch

mpl.rcParams.update({
    "font.family": "serif",
    "font.serif":  ["Computer Modern Roman", "CMU Serif", "DejaVu Serif"],
    "mathtext.fontset": "cm",
    "axes.unicode_minus": False,
    "pdf.fonttype": 42, "ps.fonttype": 42,
    "axes.linewidth": 0.8, "grid.linewidth": 0.5,
    "xtick.major.width": 0.8, "ytick.major.width": 0.8,
})

In [ ]:
# ── Output directory ───────────────────────────────────────────────────────────
IMAGE_DIR = str(Path.cwd() / "images")
os.makedirs(IMAGE_DIR, exist_ok=True)
print(IMAGE_DIR)

# ── Methods for App4 GPU runs ──────────────────────────────────────────────────
GPU_METHODS = [
    "c2qa_gpu",
    "cuda-cvdv",
    "qcvdv_scipy_gpu",
    "qcvdv_eigen_gpu",
    "qcvdv_eigen_tensor_gpu",
    "qcvdv_torch",
    "qcvdv_torch_tensor_gpu",
    "qcvdv_diaq_gpu",
]

BACKEND_DISPLAY = {
    "c2qa_gpu":               "Bosonic-Qiskit (GPU)",
    "cuda-cvdv":              "CUDA-CVDV",
    "qcvdv_scipy_gpu":        "QCVDV-SciPy (GPU)",
    "qcvdv_eigen_gpu":        "QCVDV-Eigen (GPU)",
    "qcvdv_eigen_tensor_gpu": "QCVDV-Eigen+Tensor (GPU)",
    "qcvdv_torch":            "QCVDV-Torch",
    "qcvdv_torch_tensor_gpu": "QCVDV-Torch+Tensor (GPU)",
    "qcvdv_diaq_gpu":         "QCVDV-DIAQ (GPU)",
}

# ── Okabe-Ito colorblind-safe palette ─────────────────────────────────────────
OKABE = {
    "orange": "#E69F00", "sky_blue": "#56B4E9", "bluish_green": "#009E73",
    "yellow": "#F0E442", "blue": "#0072B2", "vermillion": "#D55E00", "black": "#000000",
}

def lighten(color, amount=0.45):
    r, g, b = mcolors.to_rgb(color)
    return (r + (1-r)*amount, g + (1-g)*amount, b + (1-b)*amount)

RUN_COLORS = {
    "c2qa_gpu":               lighten(OKABE["black"],        0.50),
    "cuda-cvdv":              OKABE["blue"],
    "qcvdv_scipy_gpu":        lighten(OKABE["vermillion"],   0.45),
    "qcvdv_eigen_gpu":        lighten(OKABE["bluish_green"], 0.45),
    "qcvdv_eigen_tensor_gpu": lighten(OKABE["bluish_green"], 0.30),
    "qcvdv_torch":            lighten(OKABE["yellow"],       0.35),
    "qcvdv_torch_tensor_gpu": lighten(OKABE["sky_blue"],     0.35),
    "qcvdv_diaq_gpu":         lighten(OKABE["orange"],       0.45),
}
TRANSPILE_COLORS = {"c2qa_gpu": lighten(OKABE["black"], 0.75)}

PLOT_STYLE = {
    "axis_label_pt": 16, "ytick_pt": 13, "xtick_pt_base": 13, "legend_pt": 11,
    "min_fig_width": 7, "max_fig_width": 18, "fig_height": 5,
    "width_per_group": 0.65, "group_width": 0.82,
    "bar_edge_width": 0.7, "err_width": 0.9, "capsize": 2, "alpha": 0.88,
    "logy": True, "time_scale": 1000.0, "time_unit": "ms",
    "small_n_groups": 8, "medium_n_groups": 16, "large_n_groups": 28,
}

# ── Parse config ───────────────────────────────────────────────────────────────
# Filename format (run.sh / run_same_final.sh):
#   <benchmark>__{method}__{n_dv_qubits_N}__cv_qubits_M__lam_L__{cfg_label}.out
# split("__") → [benchmark, method, n_dv_qubits_N, cv_qubits_M, lam_L, thr, bind, hint]
_TIME_PAIR_RE = re.compile(r"([A-Za-z_]+)=([0-9eE+\-.]+)")
_TIMING_KEY_MAP = {
    "build": "build_time",     "build_time": "build_time",
    "convert": "convert_time", "convert_time": "convert_time",
    "run": "run_time",         "run_time": "run_time",
    "transpile": "transpile_time", "transpile_time": "transpile_time",
    "total": "total_time",     "total_time": "total_time",
}

def _suffix_int(t):   return int(t.split("_")[-1])
def _suffix_float(t): return float(t.split("_")[-1])

_COMMON_TIMING = ["run_time", "transpile_time", "build_time", "convert_time", "total_time"]

# app4              → run.sh          (each backend builds its own native circuit)
# app4_same_final   → run_same_final.sh (all backends share the bosonic-qiskit frontend)
_APP4_CFG = {
    "path_parsers": [_suffix_int, _suffix_int, _suffix_float],
    "path_indices":  [2, 3, 4],
    "timing_keys":   _COMMON_TIMING,
}
BENCHMARK_PARSE_CONFIG = {
    "app4":            _APP4_CFG,
    "app4_same_final": _APP4_CFG,
}

BENCHMARK_SPECS = {
    "app4":            ("CV→DV State Transfer",                ["n_dv_qubits", "cv_qubits", "lam"]),
    "app4_same_final": ("CV→DV State Transfer (Shared Circuit)", ["n_dv_qubits", "cv_qubits", "lam"]),
}

In [ ]:
# ── Parsing helpers (adapted from qcvdv_results_latest.ipynb) ─────────────────

def _parse_path_values(run_info, cfg):
    return [p(run_info[i]) for p, i in zip(cfg["path_parsers"], cfg["path_indices"])]

def _ensure_leaf(root, path_values, backend, out_file):
    cursor = root
    for v in path_values:
        cursor = cursor.setdefault(v, {})
    leaf = cursor.setdefault(backend, {})
    leaf["out_file"] = out_file
    return leaf

def _parse_timing_pairs(line):
    parsed = {}
    for raw_key, raw_value in _TIME_PAIR_RE.findall(line):
        key = _TIMING_KEY_MAP.get(raw_key)
        if key:
            parsed[key] = float(raw_value)
    return parsed

def _append_logged_timings(lines, leaf, timing_keys):
    for line in lines:
        parsed = _parse_timing_pairs(line)
        if not parsed or "ITER" not in line:
            continue
        for key in timing_keys:
            if key in parsed:
                leaf[key].append(parsed[key])

def _parse_single_run(machine_root, run_file):
    run_info = os.path.basename(run_file).replace(".out", "").split("__")
    benchmark_name = run_info[0]
    if benchmark_name not in BENCHMARK_PARSE_CONFIG:
        return
    cfg     = BENCHMARK_PARSE_CONFIG[benchmark_name]
    backend = run_info[1]
    leaf    = _ensure_leaf(machine_root, _parse_path_values(run_info, cfg), backend, run_file)
    for key in cfg["timing_keys"]:
        leaf.setdefault(key, [])
    with open(run_file) as f:
        _append_logged_timings(f.readlines(), leaf, cfg["timing_keys"])

def _resolve_raw_data_dir(path):
    for c in [Path(path), Path.cwd() / path] + [p / "raw_data" for p in Path.cwd().parents]:
        if c.exists():
            return c.resolve()
    raise FileNotFoundError(f"raw_data not found: {path!r}")

def read_more(data, RAW_DATA_DIR, MACHINE=None):
    raw_data_dir = _resolve_raw_data_dir(RAW_DATA_DIR)
    if MACHINE not in os.listdir(raw_data_dir):
        raise ValueError(f"Machine {MACHINE!r} not found in {raw_data_dir}")
    data[MACHINE] = {}
    machine_dir = os.path.join(raw_data_dir, MACHINE)
    for benchmark in os.listdir(machine_dir):
        data[MACHINE][benchmark] = {}
        for run_file in glob.glob(os.path.join(machine_dir, benchmark, "runs", "*", "*.out")):
            try:
                _parse_single_run(data[MACHINE][benchmark], run_file)
            except Exception as e:
                print(f"[WARN] {run_file}: {e}")
    return data

# ── Runtime statistics ─────────────────────────────────────────────────────────

def _is_c2qa(backend): return backend in {"c2qa", "c2qa_gpu"}

def _runtime_per_iter(entry, backend):
    if entry is None: return np.array([])
    if _is_c2qa(backend):
        t = np.array(entry.get("transpile_time", []), dtype=float)
        r = np.array(entry.get("run_time",       []), dtype=float)
        n = min(len(t), len(r))
        return t[:n] + r[:n] if n > 0 else np.array([])
    return np.array(entry.get("run_time", []), dtype=float)

def _runtime_stats(entry, backend, scale):
    vals = _runtime_per_iter(entry, backend)
    if vals.size == 0: return np.nan, 0.0, 0.0, np.nan
    total_mean = float(np.mean(vals)) * scale
    total_std  = float(np.std(vals,  ddof=0)) * scale
    if _is_c2qa(backend):
        t = np.array(entry.get("transpile_time", []), dtype=float)
        r = np.array(entry.get("run_time",       []), dtype=float)
        n = min(len(t), len(r))
        transpile_mean = float(np.mean(t[:n])) * scale if n > 0 else 0.0
        run_mean       = float(np.mean(r[:n])) * scale if n > 0 else total_mean
    else:
        transpile_mean, run_mean = 0.0, total_mean
    return total_mean, total_std, transpile_mean, run_mean

# ── Plot helpers ───────────────────────────────────────────────────────────────

def _auto_figsize(n_groups, style):
    w = max(style["min_fig_width"], min(style["max_fig_width"], style["width_per_group"] * n_groups + 3))
    return w, style["fig_height"]

def _xtick_fontsize(n_groups, style):
    base = style["xtick_pt_base"]
    if n_groups <= style["small_n_groups"]:  return base
    if n_groups <= style["medium_n_groups"]: return max(9, base - 2)
    if n_groups <= style["large_n_groups"]:  return max(8, base - 4)
    return max(7, base - 5)

def _xtick_rotation(n_groups, style):
    if n_groups <= style["small_n_groups"]:  return 0, "center"
    if n_groups <= style["medium_n_groups"]: return 22, "right"
    return 34, "right"

def _clean_val(v):
    if isinstance(v, float) and abs(v - round(v)) < 1e-12: return str(int(round(v)))
    return f"{v:g}" if isinstance(v, float) else str(v)

def _is_backend_leaf(node):
    return isinstance(node, dict) and any(b in node for b in GPU_METHODS)

def _is_timing_leaf(node):
    return isinstance(node, dict) and ("run_time" in node or "out_file" in node)

def _infer_backend(node):
    out_file = node.get("out_file", "")
    parts = Path(out_file).name.replace(".out", "").split("__") if out_file else []
    return parts[1] if len(parts) > 1 else None

def _normalize_leaf(node):
    if _is_backend_leaf(node): return node
    if _is_timing_leaf(node):
        backend = _infer_backend(node)
        return {backend: node} if backend else None
    return None

def _matches_filter(key_path, key_names, what_to_plot):
    if not what_to_plot: return True
    path_dict = dict(zip(key_names, key_path))
    for key, wanted in what_to_plot.items():
        if key not in path_dict: continue
        v = path_dict[key]
        if callable(wanted) and not wanted(v): return False
        if isinstance(wanted, (list, tuple, set)) and v not in wanted: return False
        if not callable(wanted) and not isinstance(wanted, (list, tuple, set)) and v != wanted: return False
    return True

def _collect_configs(node, key_names, what_to_plot=None, key_path=None):
    key_path = key_path or []
    norm = _normalize_leaf(node)
    if norm is not None:
        return [(tuple(key_path), norm)] if _matches_filter(tuple(key_path), key_names, what_to_plot) else []
    if not isinstance(node, dict): return []
    items = []
    for key in sorted(node):
        items.extend(_collect_configs(node[key], key_names, what_to_plot, key_path + [key]))
    return items

def _has_data(cfg, style):
    for backend in GPU_METHODS:
        total, *_ = _runtime_stats(cfg.get(backend), backend, style["time_scale"])
        if np.isfinite(total): return True
    return False

def _format_label(values, key_names, n_groups, style):
    if n_groups > style["small_n_groups"]:
        return "(" + ", ".join(_clean_val(v) for v in values) + ")"
    parts = [f"{k}={_clean_val(v)}" for k, v in zip(key_names, values)]
    return "\n".join(parts[:2])

def _save_pdf(fig, out_pdf):
    fig.savefig(out_pdf, dpi=300, transparent=True, bbox_inches="tight", pad_inches=0)


def _plot_benchmark_bars(cfg_list, labels, benchmark_name, machine, style, save_dir, show):
    pairs = [(cfg, lbl) for cfg, lbl in zip(cfg_list, labels) if _has_data(cfg, style)]
    if not pairs:
        print(f"  No data: {benchmark_name} on {machine}")
        return
    cfg_list, labels = zip(*pairs)
    n_groups   = len(labels)
    n_backends = len(GPU_METHODS)
    bar_width  = style["group_width"] / n_backends
    x          = np.arange(n_groups)
    scale      = style["time_scale"]

    fig, ax = plt.subplots(figsize=_auto_figsize(n_groups, style))
    fig.patch.set_alpha(0.0)
    ax.set_facecolor("none")

    all_tops = []
    for i, backend in enumerate(GPU_METHODS):
        pos        = x + (i - (n_backends - 1) / 2.0) * bar_width
        totals     = np.full(n_groups, np.nan)
        total_stds = np.zeros(n_groups)
        transpiles = np.zeros(n_groups)
        runs       = np.full(n_groups, np.nan)

        for j, cfg in enumerate(cfg_list):
            tm, ts, trm, rm = _runtime_stats(cfg.get(backend), backend, scale)
            totals[j], total_stds[j], transpiles[j], runs[j] = tm, ts, trm, rm

        valid = np.isfinite(totals) & np.isfinite(runs)
        if not np.any(valid): continue

        run_color = RUN_COLORS.get(backend, "#666666")
        ekw = dict(edgecolor="black", linewidth=style["bar_edge_width"])

        if _is_c2qa(backend):
            trn_color = TRANSPILE_COLORS.get(backend, lighten(run_color, 0.60))
            ax.bar(pos[valid], transpiles[valid], bar_width, color=trn_color, alpha=style["alpha"], zorder=2, **ekw)
            ax.bar(pos[valid], runs[valid],       bar_width, bottom=transpiles[valid],
                   color=run_color, alpha=style["alpha"], zorder=3, **ekw)
        else:
            ax.bar(pos[valid], runs[valid], bar_width, color=run_color, alpha=style["alpha"], zorder=2, **ekw)

        ax.errorbar(pos[valid], totals[valid], yerr=total_stds[valid],
                    fmt="none", ecolor="black", elinewidth=style["err_width"],
                    capsize=style["capsize"], zorder=5)
        all_tops.extend(totals[valid].tolist())

    if style["logy"]:
        ax.set_yscale("log")
        if all_tops:
            ax.set_ylim(bottom=max(min(all_tops) / 5.0, 1e-4), top=max(all_tops) * 3.0)

    rot, ha = _xtick_rotation(n_groups, style)
    ax.set_xticks(x)
    ax.set_xticklabels(labels, fontsize=_xtick_fontsize(n_groups, style),
                       rotation=rot, ha=ha, rotation_mode="anchor")
    ax.set_ylabel(f"Runtime ({style['time_unit']})", fontsize=style["axis_label_pt"])
    ax.tick_params(axis="y", labelsize=style["ytick_pt"])
    ax.tick_params(axis="x", pad=8)
    ax.yaxis.grid(True, which="both", alpha=0.22)
    ax.set_axisbelow(True)
    ax.margins(x=0.02)
    for spine in ("top", "right"):
        ax.spines[spine].set_visible(False)

    plt.tight_layout(pad=0.2)
    if save_dir:
        os.makedirs(save_dir, exist_ok=True)
        fname = f"{benchmark_name}_{machine}.pdf".replace(" ", "_").replace("→", "to")
        _save_pdf(fig, os.path.join(save_dir, fname))
        print(f"  → {os.path.join(save_dir, fname)}")
    if show:
        plt.show()
    else:
        plt.close(fig)


def save_legend(backends, out_pdf, ncol=None):
    display = [BACKEND_DISPLAY.get(b, b) for b in backends]
    colors  = [RUN_COLORS.get(b, "#666666") for b in backends]
    handles = [Patch(facecolor=c, edgecolor="none") for c in colors]
    ncol = ncol or len(backends)
    fig  = plt.figure(figsize=(max(2.5, 1.35 * ncol), 0.52))
    fig.patch.set_alpha(0.0)
    ax = fig.add_subplot(111); ax.axis("off")
    fig.legend(handles=handles, labels=display, loc="center", ncol=ncol,
               frameon=False, fontsize=PLOT_STYLE["legend_pt"],
               handlelength=1.1, handletextpad=0.4, columnspacing=0.8)
    _save_pdf(fig, out_pdf)
    plt.close(fig)
    print(f"  → {out_pdf}")


def plot_benchmark(data, machine, benchmark_name, what_to_plot=None,
                   save_dir=None, show=True, max_configs=None):
    save_dir = save_dir or IMAGE_DIR
    _, key_names = BENCHMARK_SPECS[benchmark_name]
    collected = _collect_configs(data[machine][benchmark_name], key_names=key_names,
                                 what_to_plot=what_to_plot)
    if not collected:
        print(f"No configs for {benchmark_name} on {machine}.")
        return
    if max_configs and len(collected) > max_configs:
        idx = np.linspace(0, len(collected)-1, max_configs, dtype=int)
        collected = [collected[i] for i in sorted(set(idx.tolist()))]
    n      = len(collected)
    labels = [_format_label(keys, key_names, n, PLOT_STYLE) for keys, _ in collected]
    cfgs   = [cfg for _, cfg in collected]
    print(f"{machine}/{benchmark_name}  ({n} configs)")
    _plot_benchmark_bars(cfgs, labels, benchmark_name, machine, PLOT_STYLE, save_dir, show)

In [ ]:
# Load all GPU data from raw_data/perlmutter_gpu/
gpu_data = read_more({}, "raw_data", MACHINE="perlmutter_gpu")
print("GPU benchmarks loaded:", sorted(gpu_data["perlmutter_gpu"].keys()))

raw_dir = _resolve_raw_data_dir("raw_data") / "perlmutter_gpu"
for bname in ["app4", "app4_same_final"]:
    n = sum(1 for _ in glob.glob(str(raw_dir / bname / "runs" / "*" / "*.out")))
    print(f"  {bname}: {n} .out files")

In [ ]:
save_legend(GPU_METHODS, os.path.join(IMAGE_DIR, "legend_gpu_app4.pdf"))

# app4: each backend uses its own native circuit construction
plot_benchmark(gpu_data, "perlmutter_gpu", "app4", show=True, max_configs=14)

# app4_same_final: all backends share the bosonic-qiskit frontend circuit
if gpu_data["perlmutter_gpu"].get("app4_same_final"):
    plot_benchmark(gpu_data, "perlmutter_gpu", "app4_same_final", show=True, max_configs=14)